In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Load dataset
print("\nLoading Dataset...")

data = pd.read_csv('student_performance_data.csv')
print(f"Loaded {len(data)} students with {len(data.columns)} columns")
print(f"\nFirst few rows:")
print(data.head(3))


Loading Dataset...
Loaded 1000 students with 12 columns

First few rows:
   total_study_hours  avg_session_length  study_sessions_per_week  \
0               17.5                88.0                     11.9   
1               14.3                78.5                     10.9   
2               18.2                61.2                     17.9   

   focus_ratio  late_night_study_ratio  days_until_deadline_avg  \
0        0.693                   0.419                      7.6   
1        0.458                   0.529                      3.2   
2        0.661                   0.130                      2.8   

   video_completion_rate  practice_problem_attempts  forum_participation  \
0                  0.754                       20.0                    1   
1                  0.596                       31.0                    1   
2                  0.432                       19.0                    1   

   study_streak_days  final_grade  pass_fail  
0                7.0        

In [3]:
#Seperate features
print("Separating features from targets...")

# Feature columns
feature_columns = [
    'total_study_hours',
    'avg_session_length', 
    'study_sessions_per_week',
    'focus_ratio',
    'late_night_study_ratio',
    'days_until_deadline_avg',
    'video_completion_rate',
    'practice_problem_attempts',
    'forum_participation',
    'study_streak_days'
]

# Extract features (X)
X = data[feature_columns].copy()

# Extract targets (y)
y_grade = data['final_grade'].copy()  # Regression target
y_pass = data['pass_fail'].copy()     # Classification target

print(f"\nFeatures (X): {X.shape}")
print(f"  - {X.shape[0]} students")
print(f"  - {X.shape[1]} features")
print(f"\nRegression target (y_grade): {y_grade.shape}")
print(f"  - Range: {y_grade.min():.1f} to {y_grade.max():.1f}")
print(f"  - Mean: {y_grade.mean():.1f}")
print(f"\nClassification target (y_pass): {y_pass.shape}")
print(f"  - Pass rate: {y_pass.mean()*100:.1f}%")

Separating features from targets...

Features (X): (1000, 10)
  - 1000 students
  - 10 features

Regression target (y_grade): (1000,)
  - Range: 45.4 to 100.0
  - Mean: 79.5

Classification target (y_pass): (1000,)
  - Pass rate: 96.9%


In [ ]:
#Train/validation/test splits
print("\n Creating training/validation/test splits...")

#* Why stratify? Ensures pass/fail ratio is similar in all splits
# This prevents having all failures in one set by chance

# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_grade_train, y_grade_temp, y_pass_train, y_pass_temp = train_test_split(
    X, y_grade, y_pass,
    test_size=0.30,      # 30% for validation + test
    random_state=42,
    stratify=y_pass      # Keep pass/fail ratio consistent
)

# Second split: Split temp into 50/50 (15% validation, 15% test of original)
X_val, X_test, y_grade_val, y_grade_test, y_pass_val, y_pass_test = train_test_split(
    X_temp, y_grade_temp, y_pass_temp,
    test_size=0.50,      # 50% of temp = 15% of original
    random_state=42,
    stratify=y_pass_temp
)

print(f"\nTraining set:   {X_train.shape[0]} students ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} students ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} students ({X_test.shape[0]/len(X)*100:.1f}%)")

# Verify stratification worked
print(f"\nPass rates are balanced:")
print(f"  - Train: {y_pass_train.mean()*100:.1f}%")
print(f"  - Val:   {y_pass_val.mean()*100:.1f}%")
print(f"  - Test:  {y_pass_test.mean()*100:.1f}%")


 Creating training/validation/test splits...

Training set:   700 students (70.0%)
Validation set: 150 students (15.0%)
Test set:       150 students (15.0%)

Pass rates are balanced:
  - Train: 96.9%
  - Val:   97.3%
  - Test:  96.7%


In [6]:
#Feature Scaling
print("Feature scalingg...")

#* Why standardize?
# - Makes features comparable (hours vs ratios)
# - Helps gradient descent converge faster
# - Required for some algorithms (SVM, neural nets)

# StandardScaler: (value - mean) / std
# Result: mean=0, std=1 for each feature
scaler = StandardScaler()

#! CRITICAL: Fit ONLY on training data
# This prevents data leakage from validation/test sets
scaler.fit(X_train)

# Transform all three sets using training(not testing data) statistics(patterns)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for easier manipulation
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X_test.index)

print("\nStandardization complete")
print(f"\nBefore scaling (training set):")
print(f"  study_hours: mean={X_train['total_study_hours'].mean():.2f}, std={X_train['total_study_hours'].std():.2f}")
print(f"  focus_ratio: mean={X_train['focus_ratio'].mean():.2f}, std={X_train['focus_ratio'].std():.2f}")

print(f"\nAfter scaling (training set):")
print(f"  study_hours: mean={X_train_scaled['total_study_hours'].mean():.2f}, std={X_train_scaled['total_study_hours'].std():.2f}")
print(f"  focus_ratio: mean={X_train_scaled['focus_ratio'].mean():.2f}, std={X_train_scaled['focus_ratio'].std():.2f}")


Feature scalingg...

Standardization complete

Before scaling (training set):
  study_hours: mean=15.06, std=4.91
  focus_ratio: mean=0.71, std=0.15

After scaling (training set):
  study_hours: mean=-0.00, std=1.00
  focus_ratio: mean=-0.00, std=1.00
